# arXiv Project — Milestones 1 & 2 Report

- **Course:** Introduction to Data Science  
- **Group:** 13  
- **Repository:** https://github.com/24127390/Intro-DS 
- **Report version/date:** 06-11-2026

## Project scope

- **Milestone 1 — Data Crawling:** resolve the assigned arXiv IDs, discover every available version, download each version's `.tex` and `.bib` source files, retrieve paper metadata and Semantic Scholar reference metadata, then analyze crawling quality and performance.
- **Milestone 2 — Hierarchical Parsing and Standardization:** follow the LaTeX compilation path, construct a hierarchy, extract and normalize elements, convert `\\bibitem` entries to BibTeX, deduplicate references and full-text elements across versions, then analyze parsing quality and performance.

Official specifications:
- [Project statement](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/)
- [Milestone 1 — Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/)
- [Milestone 2 — Parsing](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/)

## 0. Environment and reproducibility

Describe the runtime used for the final benchmark. The specification asks for a CPU-only Google Colab testbed for the final benchmarking.

- Python version: 3.11
- Environment/package versions: **TODO**
- Testbed: **TODO**
- Date/time of benchmark: **TODO**
- Dataset/code commit or version: **TODO**

In [1]:
import sys
import platform
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())

## 1. Data and implementation overview

### 1.1 Assigned ID ranges

The ranges define the input corpus (5,000 IDs per member, 25,000 in total).

| Member | Assigned month | Start suffix | End suffix | IDs | Status |
|---|---|---:|---:|---:|---|
| Hưng | 2026-03 | 10580 | 15579 | 5,000 | DONE |
| Nam | 2026-02 | 14870 | 19869 | 5,000 | DONE |
| Phong | 2026-07 | 23694 | 28693 | 5,000 | DONE |
| Toàn | 2025-12 | 03231 | 08230 | 5,000 | DONE |
| Việt | 2025-11 | 16709 | 21708 | 5,000 | DONE |

### 1.2 Pipeline architecture

1. Assigned ID resolution and version discovery (`src/crawling/source_and_references.py`: `assigned_base_ids`, `discover_assigned_versions`)
2. Metadata retrieval for every version (`crawl_arxiv_metadata`)
3. Per-version source download from the versioned `/src/` endpoint (`download_sources`)
4. Source archive extraction: only `.tex` and `.bib` are retained, figure binaries and unrelated files are excluded (`extract_tex_bib_archive`)
5. Semantic Scholar reference retrieval (`crawl_semantic_scholar`)
6. Table building: `papers`, `references` and `crawl_manifest.csv` (`src/crawling/build_tables.py`)
7. LaTeX root detection and recursive `\input` / `\include` traversal (`src/latexParser/file_resolver.py`: `LatexFileResolver`)
8. Hierarchy and element extraction, LaTeX cleanup and math normalization (`src/latexParser/structure_extractor.py`: `PublicationHierarchyBuilder`)
9. `\bibitem` to BibTeX conversion and cross-version deduplication of references and full-text elements
10. Output validation and benchmarking (`src/utils/benchmark.py`, `src/utils/table_io.py`)

**Implementation notes.**
- Every crawling stage is resumable: results are appended to CSV/JSONL manifests with `fsync`, and a rerun only processes IDs that are missing or failed.
- Requests are paced at 3.1 s, with retry and back-off on HTTP 429/5xx.
- Raw outputs live under `data/raw/` and processed Parquet tables under `data/processed/`. Neither is committed to the repository.
- Tests are in `tests/` and run with `pytest`.